# Phase 2: Fine-Tune SigLIP so400m with LoRA

**Hardware:** A10G small (24GB VRAM) — upgrade runtime before running  
**What this does:** Loads `google/siglip-so400m-patch14-384`, applies LoRA (rank=16) to
the vision encoder attention layers (~10M trainable params), trains with SigLIP's sigmoid
contrastive loss, then pushes the merged model to your Hub.

**Prerequisite:** Run `01_prepare_captions.ipynb` first.

**Before running:**
1. Upgrade runtime to A10G (Runtime → Change runtime type → A10G)
2. Add secrets via the key icon (🔑) in the left sidebar — enable notebook access for each:
   - `HF_TOKEN` — HF token with write access
   - `HF_USERNAME` — your HF username
3. Fill in the form fields in Cell 2:
   - Click **▶ Run** on Cell 2 — a form appears **above** the code
   - Set `SOURCE_DATASET` to your captioned dataset (output of Notebook 1)
   - Set `OUTPUT_REPO` to where the fine-tuned model should be saved on HF Hub
   - Adjust `BATCH_SIZE`, `GRAD_ACCUM`, `NUM_EPOCHS`, `LORA_RANK` if needed
4. Run remaining cells top to bottom with **Shift+Enter**

**If OOM:** Set `BATCH_SIZE=4` and `GRAD_ACCUM=8` in the Cell 2 form.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets peft accelerate Pillow torch huggingface-hub

In [ ]:
# Cell 2 — Config
# @title Configuration
# @markdown Fill in the dataset and output details below, then run this cell.
from google.colab import userdata
from huggingface_hub import login

# Secrets (set in the key icon in left sidebar)
HF_TOKEN    = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')

# Authenticate — required for gated datasets and pushing to Hub
login(token=HF_TOKEN)

# Form variables — edit these before running
SOURCE_DATASET = "your-username/output_ds" # @param {type:"string"}
OUTPUT_REPO    = "your-username/output_model" # @param {type:"string"}

# Training hyperparameters — adjust if needed
BATCH_SIZE    = 8    # @param {type:"integer"}
GRAD_ACCUM    = 4    # @param {type:"integer"}
NUM_EPOCHS    = 3    # @param {type:"integer"}
LORA_RANK     = 16   # @param {type:"integer"}

# Fixed values
BASE_MODEL    = "google/siglip-so400m-patch14-384"
LEARNING_RATE = 2e-4
WARMUP_STEPS  = 50
MAX_LENGTH    = 64

# Verify
print("HF_TOKEN:    ", HF_TOKEN[:8] + "..." if HF_TOKEN else "ERROR: not set")
print("HF_USERNAME: ", HF_USERNAME if HF_USERNAME else "ERROR: not set")
print("SOURCE:      ", SOURCE_DATASET)
print("OUTPUT_REPO: ", OUTPUT_REPO)
print(f"Training:     {NUM_EPOCHS} epochs, batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} effective")

In [ ]:
# Cell 3 — Load model + apply LoRA to vision encoder
import torch
from transformers import AutoProcessor, AutoModel
from peft import get_peft_model, LoraConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

processor = AutoProcessor.from_pretrained(BASE_MODEL)
model     = AutoModel.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)

# LoRA on vision encoder attention only (~10M trainable params)
model.vision_model = get_peft_model(
    model.vision_model,
    LoraConfig(r=LORA_RANK, lora_alpha=32, lora_dropout=0.1,
               bias="none", target_modules=["q_proj", "v_proj"])
)
model.vision_model.print_trainable_parameters()

# Freeze text encoder entirely
for p in model.text_model.parameters():
    p.requires_grad = False

# Keep logit scale and bias trainable
model.logit_scale.requires_grad = True
model.logit_bias.requires_grad  = True

model = model.to(device).train()
print("Model ready")

In [ ]:
# Cell 4 — DataLoader
from torch.utils.data import DataLoader
from datasets import load_dataset

ds = load_dataset(SOURCE_DATASET, split="train", token=HF_TOKEN)
print(f"{len(ds)} pairs loaded")

def collate(batch):
    inp = processor(
        text=[b["caption"] for b in batch],
        images=[b["image"].convert("RGB") for b in batch],
        return_tensors="pt", padding="max_length",
        max_length=MAX_LENGTH, truncation=True,
    )
    return {k: v.to(device) for k, v in inp.items()}

loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate, num_workers=2)
print(f"{len(loader)} batches per epoch")

In [ ]:
# Cell 5 — SigLIP sigmoid loss
import torch.nn.functional as F

def siglip_loss(img_emb, txt_emb, logit_scale, logit_bias):
    """SigLIP sigmoid contrastive loss. +1 for diagonal pairs, -1 for off-diagonal."""
    img = F.normalize(img_emb.float(), dim=-1)
    txt = F.normalize(txt_emb.float(), dim=-1)
    logits = torch.matmul(img, txt.T) * logit_scale.exp() + logit_bias
    n      = logits.shape[0]
    labels = 2 * torch.eye(n, device=logits.device) - 1
    return -F.logsigmoid(labels * logits).mean()

In [ ]:
# Cell 6 — Optimizer + scheduler
import math

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.01)
total_steps = (len(loader) // GRAD_ACCUM) * NUM_EPOCHS

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: (s / max(1, WARMUP_STEPS)) if s < WARMUP_STEPS
              else max(0.0, 0.5 * (1 + math.cos(math.pi *
                       (s - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS))))
)
print(f"Training for {total_steps} optimizer steps ({NUM_EPOCHS} epochs)")

In [ ]:
# Cell 7 — Training loop
import tempfile
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

def save_and_push(tag="update"):
    """Merge LoRA adapters into base weights and push to Hub."""
    model.vision_model = model.vision_model.merge_and_unload()
    with tempfile.TemporaryDirectory() as d:
        model.save_pretrained(d)
        processor.save_pretrained(d)
        api.upload_folder(folder_path=d, repo_id=OUTPUT_REPO,
                          repo_type="model", commit_message=tag, token=HF_TOKEN)
    print(f"  Pushed to {OUTPUT_REPO} ({tag})")

global_step = 0
best_loss   = float("inf")
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for step, batch in enumerate(loader):
        img_emb = model.get_image_features(pixel_values=batch["pixel_values"])
        txt_emb = model.get_text_features(input_ids=batch["input_ids"],
                                          attention_mask=batch["attention_mask"])
        loss = siglip_loss(img_emb, txt_emb,
                           model.logit_scale, model.logit_bias) / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
            epoch_loss  += loss.item() * GRAD_ACCUM

            if global_step % 10 == 0:
                print(f"Ep {epoch+1} Step {global_step}/{total_steps} "
                      f"Loss {loss.item()*GRAD_ACCUM:.4f}")

    avg = epoch_loss / max(1, len(loader) // GRAD_ACCUM)
    print(f"\nEpoch {epoch+1} avg loss: {avg:.4f}")
    if avg < best_loss:
        best_loss = avg
        save_and_push(tag=f"best-epoch-{epoch+1}")

save_and_push(tag="final")
print("Training complete.")